# 07 · Deduplicar una base de IMPORTADORES por RAZÓN SOCIAL + PAÍS

**Qué hace.** Toma una base de destinatarios de exportación —una fila por
`(razón social, país)`— y devuelve **una correlativa**: cada nombre y país
originales con el **nombre final** y el **país final** que se les asignó, más
las métricas que permiten juzgar si ese emparejamiento es creíble.

**Variables de empalme:** razón social y país. Dos registros solo pueden
unirse si el país canónico coincide **exactamente** y el nombre supera el
umbral de similitud. El país es, además, un bloqueo duro: nunca se fusionan
registros de países distintos.

**Cómo usarlo (4 pasos)**
1. Ejecute la **Celda A** (entorno; una vez por sesión).
2. Edite la **Celda B** (rutas, columnas y parámetros). Es la celda principal.
3. Revise las **Celdas C y D** si quiere ajustar el catálogo de países o las
   listas de términos a limpiar. Traen valores por defecto que funcionan.
4. Ejecute **E → F → G → H** y descargue `/content/resultados/`.

---

## Lo que este notebook entiende del problema

Deduplicar razones sociales de comercio exterior no es comparar cadenas. Tres
modos de falla, los tres medidos sobre la base real de 211.949 registros:

| Modo de falla | Ejemplo real | Qué lo causa | Cómo se ataca aquí |
|---|---|---|---|
| **Imán genérico** | `INTERNATIONAL` absorbía 157 razones sociales distintas | un nombre corto está "contenido" en todos los que lo mencionan | la contención exige compartir al menos un token **no genérico** y distintivo |
| **Prefijo compartido** | `COMERCIALIZADORA ATLANTA C.A.` vs `COMERCIALIZADORA ATLANTIC C.A.` | Jaro-Winkler premia el prefijo común, y el prefijo es la parte genérica | ponderación **IDF**: el token que decide es el informativo |
| **Encadenamiento** | `a≈b`, `b≈c`, pero `a≉c` | el clustering por componentes conexas (single-linkage) propaga | **cobertura por estrellas**: todo miembro queda a ≤ umbral de SU nombre final |

Y un cuarto que no es de nombres sino de país: `TURQUÍA`/`TÜRKIYE`,
`CHEQUIA`/`REPÚBLICA CHECA`, `YIBUTI`/`DJIBOUTI`, `COSTA DE MARFIL`/`CÔTE
D'IVOIRE`. **Ninguna similitud de cadenas resuelve esos pares.** El país se
canoniza contra un **catálogo declarado** (Celda C), no por fuzzy matching.

> Motor: `rues-linker 0.21.0` — `EsquemaCampos`, `evaluar_esquema`,
> `clusters_desde_decisiones`, `LSHTexto`, `construir_idf`, `normalizar_nombre`.
> Este notebook añade, vía el punto de extensión `CampoSpec(comparador=...)`,
> un comparador de nombres IDF + Jaro-Winkler y un refinamiento por cobertura.
>
> En Colab Free suba los archivos a `/content/datos` y descargue
> `/content/resultados` antes de cerrar: el disco local es efímero.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  CELDA A — ENTORNO   (ejecutar UNA vez por sesión; normalmente no se edita)
# ══════════════════════════════════════════════════════════════════════════
from __future__ import annotations

import importlib
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

VERSION_OBJETIVO = "0.21.0"

# Si el tag público de GitHub aún no es 0.21.0, apunte aquí a su copia local:
# un .whl, un .zip del proyecto o la carpeta descomprimida. Déjelo en "" para
# instalar desde GitHub.
RUTA_PAQUETE_LOCAL = ""  # p. ej. "/content/rues_linker_pruebas"

ORIGEN_GIT = f"rues-linker @ git+https://github.com/EnriqueForero/rues-linker@v{VERSION_OBJETIVO}"


def _t(x: str) -> tuple[int, ...]:
    base = x.split("+")[0].split("-")[0]
    return tuple(int(p) for p in base.split(".")[:3] if p.isdigit())


def _version_ok(v: str) -> bool:
    try:
        return _t(v) == _t(VERSION_OBJETIVO)
    except Exception:
        return False


def preparar_entorno() -> None:
    """Garantiza rues-linker 0.21.0 y falla con un mensaje accionable si no."""
    try:
        instalada = version("rues-linker")
    except PackageNotFoundError:
        instalada = None
    if instalada is not None and _version_ok(instalada):
        import record_linkage

        if not _version_ok(record_linkage.__version__):
            raise RuntimeError(
                "Qué pasó: hay otra versión de record_linkage cargada en memoria.\n"
                "Por qué importa: el motor y el notebook dejarían de coincidir.\n"
                "Qué hacer: Entorno de ejecución → Reiniciar sesión, y repita esta celda."
            )
        print(f"rues-linker {record_linkage.__version__} ya instalado — listo")
        return

    origen = RUTA_PAQUETE_LOCAL.strip() or ORIGEN_GIT
    print(f"Instalando rues-linker desde: {origen}")
    r = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", origen],
        capture_output=True,
        text=True,
    )
    if r.returncode != 0:
        raise RuntimeError(
            "Qué pasó: no fue posible instalar rues-linker.\n"
            "Por qué importa: sin el motor el notebook no puede correr.\n"
            "Qué hacer: si el tag v"
            + VERSION_OBJETIVO
            + " todavía no está publicado en GitHub, suba el .zip del "
            "proyecto a /content y ponga su ruta en RUTA_PAQUETE_LOCAL.\n"
            f"  Detalle de pip: {r.stderr.strip()[-500:]}"
        )
    importlib.invalidate_caches()
    if "record_linkage" in sys.modules:
        raise RuntimeError(
            "Qué pasó: se instaló la librería pero una versión anterior sigue cargada.\n"
            "Qué hacer: Entorno de ejecución → Reiniciar sesión, y repita esta celda."
        )
    import record_linkage

    if not _version_ok(record_linkage.__version__):
        raise RuntimeError(
            f"Qué pasó: quedó instalada {record_linkage.__version__} y se requiere "
            f"{VERSION_OBJETIVO}.\nQué hacer: reinicie el entorno y repita esta celda."
        )
    print(f"rues-linker {record_linkage.__version__} instalado — listo")


preparar_entorno()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  CELDA B — CONFIGURACIÓN        (LA CELDA PRINCIPAL QUE USTED EDITA)
# ══════════════════════════════════════════════════════════════════════════
from dataclasses import dataclass, field

# ── 1. Rutas ────────────────────────────────────────────────────────────────
#  En Colab: suba el archivo con el panel "Archivos" a /content/datos/.
#  Fuera de Colab: ponga rutas absolutas de su máquina.
RUTA_ENTRADA = "/content/datos/empresas_importadoras.csv"   # @param {type:"string"}
CARPETA_SALIDA = "/content/resultados/07_importadores"      # @param {type:"string"}
HOJA_EXCEL = 0          # solo si RUTA_ENTRADA es .xlsx
SEPARADOR_CSV = ","     # "," | ";" | "\t"
ENCODING_CSV = "utf-8-sig"

# ── 2. Columnas de SU archivo ───────────────────────────────────────────────
COL_RAZON_SOCIAL = "RAZON_SOCIAL_DESTINATARIO"   # @param {type:"string"}
COL_PAIS = "PAIS_DESTINO_FINAL"                  # @param {type:"string"}
#  Columnas numéricas que se suman por grupo (opcionales; [] si no tiene).
COLS_METRICAS = ["NUM_REGISTROS_EXPORTACION", "VALOR_FOB_USD_TOTAL"]
#  Cuál de ellas manda al elegir el nombre final cuando hay empate de frecuencia.
COL_PESO_ECONOMICO = "VALOR_FOB_USD_TOTAL"       # "" si no aplica


@dataclass
class Config:
    """Parámetros del emparejamiento. Todos medidos, ninguno inventado.

    Los valores por defecto son los de la corrida documentada sobre
    211.949 registros de destinatarios de exportación colombianos.
    """

    # ── Umbral de decisión ──────────────────────────────────────────────────
    #: Similitud mínima de nombre para unir dos registros, en escala 0–1.
    #: Es LA perilla. Medido sobre 140 asignaciones revisadas a mano:
    #:   0.84 → precisión 0.94 (defecto, equilibrado)
    #:   0.88 → precisión 0.97, pero pierde ~7.000 empalmes correctos
    #:   0.80 → gana cobertura con precisión ~0.75 en el tramo añadido
    umbral_nombre: float = 0.84

    #: Cuánto pesa la evidencia de TOKENS (IDF) frente a la de CADENA (Jaro-
    #: Winkler): sim = alfa·tokens + (1−alfa)·JW. Subir alfa castiga más los
    #: prefijos genéricos; bajarlo perdona más erratas de digitación.
    alfa_idf: float = 0.25

    #: Peso del prefijo común en Jaro-Winkler (0–0.25). 0.10 es el estándar.
    prefix_weight: float = 0.10

    # ── Regla de contención (nombre corto dentro de nombre largo) ───────────
    #: Cuántos tokens INFORMATIVOS pueden diferir para que "A está contenido
    #: en B" cuente como evidencia. 0 = lo que sobra debe ser ruido declarado
    #: (genéricos, números, siglas). Subirlo fusiona consolidadores con sus
    #: clientes finales ("MQE ‹cliente›" → "MQE"): mídalo antes.
    max_diferencia_informativa: int = 0

    #: Mínimo de tokens informativos que el par debe compartir.
    min_informativos_compartidos: int = 1

    #: El token compartido más distintivo debe aparecer, como mucho, en este
    #: número de nombres del corpus. Evita que una palabra de sector
    #: ("FLOWERS") haga las veces de identidad.
    frecuencia_max_distintivo: int = 1000

    #: Longitud mínima de un token para considerarlo informativo.
    longitud_minima_token: int = 3

    #: Tokens puramente numéricos (guías, códigos de cliente) no identifican
    #: a una empresa: identifican un envío. Se excluyen por regla.
    ignorar_numericos: bool = True

    #: Peso al que se baja el IDF de un término declarado genérico.
    peso_token_generico: float = 1.0

    # ── Bloqueo (qué pares se llegan a comparar) ────────────────────────────
    #: Medido por fuerza bruta sobre 5 particiones completas — recall del
    #: bloqueo (PC = pares verdaderos recuperados):
    #:   perm=128, umbral=0.25 → PC 0.99–1.00   (defecto)
    #:   perm= 64, umbral=0.30 → PC 0.97–0.99   (más rápido)
    #:   perm= 64, umbral=0.35 → PC 0.67–0.81   ← NO usar: pierde 1 de cada 4
    lsh_permutaciones: int = 128
    lsh_umbral: float = 0.25
    lsh_ngram: int = 3
    lsh_max_grupo: int = 500
    max_candidatos: int = 60_000_000

    # ── Score del esquema multicampo ────────────────────────────────────────
    peso_nombre: float = 2.0
    peso_pais: float = 1.0

    # ── Postproceso ─────────────────────────────────────────────────────────
    #: Corta el encadenamiento transitivo. Déjelo en True salvo que quiera
    #: reproducir el single-linkage clásico.
    refinar_cohesion: bool = True

    #: Regla para elegir el nombre final dentro de un grupo.
    #: "cobertura"  → el que más miembros cubre (medoide operativo; defecto)
    #: "frecuencia" → el que aparece en más filas del archivo original
    regla_nombre_final: str = "cobertura"

    # ── Control de calidad ──────────────────────────────────────────────────
    #: Particiones (ISO3) sobre las que medir el recall del bloqueo por fuerza
    #: bruta. Vacío = no medir. Cuidado: el costo es O(k²) por partición.
    paises_auditoria_bloqueo: tuple[str, ...] = ("GTM", "NLD", "DEU")
    #: Tamaño de la muestra estratificada que se exporta para revisión manual.
    muestra_revision_por_banda: int = 40
    semilla: int = 42

    def __post_init__(self) -> None:
        if not 0.0 < self.umbral_nombre < 1.0:
            raise ValueError(f"umbral_nombre={self.umbral_nombre} fuera de (0, 1).")
        if not 0.0 <= self.alfa_idf <= 1.0:
            raise ValueError(f"alfa_idf={self.alfa_idf} fuera de [0, 1].")
        if self.regla_nombre_final not in {"cobertura", "frecuencia"}:
            raise ValueError("regla_nombre_final: 'cobertura' | 'frecuencia'.")

    @property
    def sim_minima(self) -> float:
        """Umbral traducido a la escala firmada [-1, 1] del motor."""
        return 2.0 * self.umbral_nombre - 1.0

    @property
    def umbral_score(self) -> float:
        """Umbral del score combinado, calculado para que EQUIVALGA al de nombre.

        Dentro de una partición el país siempre concuerda (similitud 1.0), así
        que fijar aquí el valor exacto hace que ``score ≥ umbral_score`` y
        ``similitud_nombre ≥ umbral_nombre`` sean la MISMA condición. Sin este
        cálculo, el umbral de score sería un número decorativo.
        """
        return ((self.peso_nombre * self.sim_minima + self.peso_pais)
                / (self.peso_nombre + self.peso_pais))


CFG = Config()
print(f"umbral de nombre {CFG.umbral_nombre}  →  umbral de score {CFG.umbral_score:.4f}")
print(f"bloqueo LSH: {CFG.lsh_permutaciones} permutaciones @ {CFG.lsh_umbral}")

## Celdas C y D — los dos diccionarios editables

El catálogo de países y las listas de limpieza son **datos**, no código: viven en celdas propias para que se puedan editar, versionar y auditar sin tocar el motor.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  CELDA C — CATÁLOGO DE PAÍSES   (edítelo para añadir alias o países)
# ══════════════════════════════════════════════════════════════════════════
#
#  Por qué un catálogo y no similitud de cadenas: en la base real conviven
#  TURQUIA/TURKIYE, CHEQUIA/REPUBLICA CHECA, YIBUTI/DJIBOUTI y COSTA DE
#  MARFIL/COTE D'IVOIRE. Ningún comparador de texto une esos pares, y varios
#  pares que SÍ se parecen —GUINEA / GUINEA ECUATORIAL / GUINEA-BISSAU,
#  CONGO / REPUBLICA DEMOCRATICA DEL CONGO— son países distintos. Es un
#  problema de diccionario, y el diccionario es un dato versionado.
#
#  Formato: (ISO 3166-1 alfa-3, NOMBRE CANÓNICO, (alias adicionales, ...)).
#  Los alias se comparan sin tildes, en mayúsculas y sin puntuación, así que
#  no hace falta declarar variantes de acentuación ni de puntuación.
#  Añadir un país: agregue una fila. Añadir una grafía: agréguela a la tupla.

CATALOGO_PAISES: tuple[tuple[str, str, tuple[str, ...]], ...] = (
    ("USA", "ESTADOS UNIDOS", ("ESTADOS UNIDOS DE AMERICA", "EEUU", "EE UU", "USA", "UNITED STATES", "UNITED STATES OF AMERICA", "US")),
    ("ECU", "ECUADOR", ()),
    ("PAN", "PANAMA", ()),
    ("PER", "PERU", ()),
    ("MEX", "MEXICO", ()),
    ("VEN", "VENEZUELA", ("VENEZUELA (REPUBLICA BOLIVARIANA DE)", "REPUBLICA BOLIVARIANA DE VENEZUELA")),
    ("DOM", "REPUBLICA DOMINICANA", ("REP DOMINICANA",)),
    ("CAN", "CANADA", ()),
    ("CRI", "COSTA RICA", ()),
    ("ESP", "ESPANA", ("SPAIN",)),
    ("GTM", "GUATEMALA", ()),
    ("CHL", "CHILE", ()),
    ("BRA", "BRASIL", ("BRAZIL",)),
    ("NLD", "PAISES BAJOS", ("PAISES BAJOS (REINO DE LOS)", "HOLANDA", "NETHERLANDS")),
    ("SLV", "EL SALVADOR", ("SALVADOR",)),
    ("HND", "HONDURAS", ()),
    ("PRI", "PUERTO RICO", ()),
    ("DEU", "ALEMANIA", ("GERMANY", "REPUBLICA FEDERAL DE ALEMANIA")),
    ("CHN", "CHINA", ("REPUBLICA POPULAR CHINA", "CHINA CONTINENTAL")),
    ("KAZ", "KAZAJSTAN", ("KAZAJISTAN", "KAZAKHSTAN")),
    ("ARG", "ARGENTINA", ()),
    ("GBR", "REINO UNIDO", ("REINO UNIDO DE GRAN BRETANA E IRLANDA DEL NORTE", "GRAN BRETANA", "INGLATERRA", "UNITED KINGDOM")),
    ("ARE", "EMIRATOS ARABES UNIDOS", ("EAU",)),
    ("ITA", "ITALIA", ()),
    ("JPN", "JAPON", ()),
    ("BEL", "BELGICA", ()),
    ("FRA", "FRANCIA", ()),
    ("NIC", "NICARAGUA", ()),
    ("ABW", "ARUBA", ()),
    ("CUW", "CURAZAO", ("CURACAO",)),
    ("BOL", "BOLIVIA", ("BOLIVIA (ESTADO PLURINACIONAL DE)", "ESTADO PLURINACIONAL DE BOLIVIA")),
    ("POL", "POLONIA", ()),
    ("CUB", "CUBA", ()),
    ("IND", "INDIA", ()),
    ("AUS", "AUSTRALIA", ()),
    ("JAM", "JAMAICA", ()),
    ("KOR", "COREA DEL SUR", ("COREA (SUR) REPUBLICA DE", "COREA (SUR) REP DE", "REPUBLICA DE COREA", "COREA SUR", "COREA")),
    ("URY", "URUGUAY", ()),
    ("SAU", "ARABIA SAUDITA", ("ARABIA SAUDI",)),
    ("PRY", "PARAGUAY", ()),
    ("RUS", "RUSIA", ("FEDERACION DE RUSIA", "FEDERACION RUSA")),
    ("TWN", "TAIWAN", ("TAIWAN (PROVINCIA DE CHINA)", "FORMOSA")),
    ("TTO", "TRINIDAD Y TOBAGO", ("TRINIDAD Y TABAGO",)),
    ("HKG", "HONG KONG", ("CHINA, REGION ADMINISTRATIVA ESPECIAL DE HONG KONG", "CHINA REGION ADMINISTRATIVA ESPECIAL DE HONG KONG", "HONG KONG (CHINA)")),
    ("CHE", "SUIZA", ()),
    ("GUY", "GUYANA", ()),
    ("TUR", "TURQUIA", ("TURKIYE", "TURKEY")),
    ("PRT", "PORTUGAL", ()),
    ("ISR", "ISRAEL", ()),
    ("THA", "TAILANDIA", ()),
    ("MYS", "MALASIA", ("MALAYSIA",)),
    ("SGP", "SINGAPUR", ("SINGAPORE",)),
    ("QAT", "QATAR", ("CATAR",)),
    ("HTI", "HAITI", ()),
    ("KWT", "KUWAIT", ()),
    ("CZE", "CHEQUIA", ("REPUBLICA CHECA",)),
    ("BRB", "BARBADOS", ()),
    ("EGY", "EGIPTO", ()),
    ("GRC", "GRECIA", ()),
    ("IDN", "INDONESIA", ()),
    ("VNM", "VIET NAM", ("VIETNAM",)),
    ("SUR", "SURINAME", ("SURINAM",)),
    ("SWE", "SUECIA", ()),
    ("ROU", "RUMANIA", ("RUMANIA (RUMANIA)", "ROMANIA")),
    ("BHS", "BAHAMAS", ()),
    ("BLZ", "BELICE", ("BELIZE",)),
    ("HUN", "HUNGRIA", ()),
    ("OMN", "OMAN", ()),
    ("NOR", "NORUEGA", ()),
    ("AUT", "AUSTRIA", ()),
    ("UKR", "UCRANIA", ()),
    ("PHL", "FILIPINAS", ()),
    ("DNK", "DINAMARCA", ()),
    ("GLP", "GUADALUPE", ()),
    ("IRL", "IRLANDA", ("IRLANDA (EIRE)", "EIRE")),
    ("NGA", "NIGERIA", ()),
    ("LBN", "LIBANO", ()),
    ("NZL", "NUEVA ZELANDA", ("NUEVA ZELANDIA",)),
    ("KGZ", "KIRGUISTAN", ("KIRGUIZISTAN", "KYRGYZSTAN")),
    ("LCA", "SANTA LUCIA", ()),
    ("EST", "ESTONIA", ()),
    ("BHR", "BAHREIN", ()),
    ("JOR", "JORDANIA", ()),
    ("UZB", "UZBEKISTAN", ()),
    ("ZAF", "SUDAFRICA", ("SUDAFRICA, REPUBLICA DE", "SUDAFRICA REPUBLICA DE", "REPUBLICA DE SUDAFRICA")),
    ("FIN", "FINLANDIA", ()),
    ("MAR", "MARRUECOS", ()),
    ("MTQ", "MARTINICA", ()),
    ("BLR", "BELARUS", ("BIELORRUSIA",)),
    ("SVN", "ESLOVENIA", ()),
    ("BGR", "BULGARIA", ()),
    ("LBY", "LIBIA", ()),
    ("LVA", "LETONIA", ()),
    ("CYM", "ISLAS CAIMAN", ("CAIMAN ISLAS", "CAIMAN, ISLAS")),
    ("AZE", "AZERBAIYAN", ("AZERBAIJAN",)),
    ("GRD", "GRANADA", ("GRENADA",)),
    ("ATG", "ANTIGUA Y BARBUDA", ()),
    ("GHA", "GHANA", ()),
    ("PAK", "PAKISTAN", ()),
    ("GEO", "GEORGIA", ()),
    ("IRQ", "IRAQ", ("IRAK",)),
    ("DZA", "ARGELIA", ()),
    ("SVK", "ESLOVAQUIA", ()),
    ("CIV", "COSTA DE MARFIL", ("COTE D'IVOIRE", "COTE DIVOIRE", "COTE D IVOIRE")),
    ("GAB", "GABON", ()),
    ("VCT", "SAN VICENTE Y LAS GRANADINAS", ()),
    ("DMA", "DOMINICA", ()),
    ("HRV", "CROACIA", ()),
    ("AGO", "ANGOLA", ()),
    ("ARM", "ARMENIA", ()),
    ("GUF", "GUAYANA FRANCESA", ()),
    ("LTU", "LITUANIA", ()),
    ("TCA", "ISLAS TURCAS Y CAICOS", ("TURCAS Y CAICOS, ISLAS", "TURCAS Y CAICOS ISLAS")),
    ("MNE", "MONTENEGRO", ()),
    ("KEN", "KENIA", ("KENYA",)),
    ("LKA", "SRI LANKA", ()),
    ("TGO", "TOGO", ()),
    ("CMR", "CAMERUN", ("CAMERUN REP UNIDA DEL", "REPUBLICA UNIDA DEL CAMERUN")),
    ("ETH", "ETIOPIA", ()),
    ("SEN", "SENEGAL", ()),
    ("COG", "CONGO", ("CONGO REPUBLICA DEL", "REPUBLICA DEL CONGO", "CONGO BRAZZAVILLE")),
    ("MDA", "MOLDOVA", ("REPUBLICA DE MOLDOVA", "MOLDAVIA")),
    ("LBR", "LIBERIA", ()),
    ("UGA", "UGANDA", ()),
    ("ALB", "ALBANIA", ()),
    ("DJI", "DJIBOUTI", ("YIBUTI",)),
    ("COD", "REPUBLICA DEMOCRATICA DEL CONGO", ("REPUBDEMOCRATICA DEL CONGO", "CONGO KINSHASA", "ZAIRE")),
    ("BIH", "BOSNIA Y HERZEGOVINA", ()),
    ("MLT", "MALTA", ()),
    ("COL", "COLOMBIA", ()),
    ("TUN", "TUNEZ", ("TUNICIA",)),
    ("CYP", "CHIPRE", ()),
    ("SRB", "SERBIA", ()),
    ("VGB", "ISLAS VIRGENES BRITANICAS", ("VIRGENES ISLAS (BRITANICAS)", "VIRGENES ISLAS (REINO UNIDO)", "ISLAS VIRGENES (BRITANICAS)")),
    ("MOZ", "MOZAMBIQUE", ()),
    ("BMU", "BERMUDAS", ("BERMUDA",)),
    ("TZA", "TANZANIA", ("TANZANIA, REPUBLICA UNIDA DE", "TANZANIA REP UNIDA DE", "REPUBLICA UNIDA DE TANZANIA")),
    ("GIN", "GUINEA", ()),
    ("LUX", "LUXEMBURGO", ()),
    ("BGD", "BANGLADESH", ()),
    ("GNQ", "GUINEA ECUATORIAL", ()),
    ("KNA", "SAN CRISTOBAL Y NIEVES", ("SAINT KITTS Y NEVIS", "SANCRISTOBAL Y NIEVES", "SAN CRISTOBAL Y NEVIS")),
    ("MDG", "MADAGASCAR", ()),
    ("MUS", "MAURICIO", ()),
    ("CPV", "CABO VERDE", ()),
    ("VIR", "ISLAS VIRGENES DE LOS ESTADOS UNIDOS", ("VIRGENES ISLAS (ESTADOS UNIDO", "VIRGENES ISLAS (ESTADOS UNIDOS)", "ISLAS VIRGENES (EEUU)")),
    ("GUM", "GUAM", ()),
    ("ZWE", "ZIMBABWE", ()),
    ("NCL", "NUEVA CALEDONIA", ()),
    ("MDV", "MALDIVAS", ()),
    ("MKD", "MACEDONIA DEL NORTE", ("MACEDONIA",)),
    ("MLI", "MALI", ()),
    ("NAM", "NAMIBIA", ()),
    ("BEN", "BENIN", ()),
    ("KHM", "CAMBOYA", ("CAMBOYA (KAMPUCHEA)", "KAMPUCHEA")),
    ("SDN", "SUDAN", ()),
    ("MRT", "MAURITANIA", ()),
    ("ZMB", "ZAMBIA", ()),
    ("ISL", "ISLANDIA", ()),
    ("BRN", "BRUNEI DARUSSALAM", ("BRUNEI",)),
    ("TJK", "TAYIKISTAN", ("TADJIKISTAN",)),
    ("YEM", "YEMEN", ()),
    ("AIA", "ANGUILA", ("ANGUILLA",)),
    ("SLE", "SIERRA LEONA", ()),
    ("MAC", "MACAO", ("CHINA, REGION ADMINISTRATIVA ESPECIAL DE MACAO", "CHINA REGION ADMINISTRATIVA ESPECIAL DE MACAO")),
    ("MCO", "MONACO", ()),
    ("TCD", "CHAD", ()),
    ("PYF", "POLINESIA FRANCESA", ()),
    ("PSE", "ESTADO DE PALESTINA", ("PALESTINA",)),
    ("SYR", "SIRIA", ("SIRIA, REPUBLICA ARABE DE", "SIRIA REP ARABE DE", "REPUBLICA ARABE SIRIA")),
    ("REU", "REUNION", ()),
    ("PRK", "COREA DEL NORTE", ("REPUBLICA POPULAR DEMOCRATICA DE COREA", "COREA (NORTE) REPPOPDEM DE", "COREA NORTE")),
    ("COM", "COMORAS", ()),
    ("GNB", "GUINEA-BISSAU", ("GUINEA BISSAU", "GUINEA - BISSAU")),
    ("MNG", "MONGOLIA", ()),
    ("BFA", "BURKINA FASO", ()),
    ("SYC", "SEYCHELLES", ()),
    ("CAF", "REPUBLICA CENTROAFRICANA", ()),
    ("PNG", "PAPUA NUEVA GUINEA", ("PAPUASIA NUEVA GUINEA",)),
    ("RWA", "RUANDA", ("RWANDA",)),
    ("MSR", "MONTSERRAT", ("MONTSERRAT ISLA",)),
    ("FJI", "FIJI", ("FIDJI",)),
    ("IOT", "TERRITORIO BRITANICO DEL OCEANO INDICO", ("T BRIT OCEANO INDICO",)),
    ("BES", "BONAIRE, SAN EUSTAQUIO Y SABA", ()),
    ("GMB", "GAMBIA", ()),
    ("SWZ", "ESWATINI", ("SUAZILANDIA",)),
    ("BDI", "BURUNDI", ()),
    ("BWA", "BOTSWANA", ("BOTSUANA",)),
    ("TON", "TONGA", ()),
    ("AND", "ANDORRA", ()),
    ("FSM", "MICRONESIA", ("MICRONESIA (ESTADOS FEDERADOS DE)",)),
    ("SOM", "SOMALIA", ()),
    ("MHL", "ISLAS MARSHALL", ("MARSHALL, ISLAS",)),
    ("NER", "NIGER", ()),
    ("NPL", "NEPAL", ()),
    ("LSO", "LESOTHO", ()),
    ("TLS", "TIMOR ORIENTAL", ("TIMOR-LESTE",)),
    ("SMR", "SAN MARINO", ()),
    ("LAO", "LAOS", ("LAO, REPUBLICA DEMOCRATICA POPULAR DE", "LAOS REPPOP DEMOCRATICA DE")),
    ("MMR", "MYANMAR", ("BIRMANIA",)),
    ("TKM", "TURKMENISTAN", ()),
    ("ASM", "SAMOA AMERICANA", ()),
    ("MWI", "MALAWI", ()),
)

#  Valores que NO son un país sino una zona franca colombiana. En la base real
#  son 76 grafías / 397 filas cuyo "país de destino final" es un usuario de
#  zona franca. Se agrupan en un único valor y se marcan, porque mezclarlos
#  con destinos reales contamina cualquier análisis por mercado.
PATRONES_NO_PAIS: tuple[str, ...] = (r"^ZONA\s+FRANCA\b", r"^ZFPE?\b", r"^ZF\b")
ETIQUETA_NO_PAIS = "ZONA FRANCA (COLOMBIA)"
ISO_NO_PAIS = "ZZF"

print(f"catálogo: {len(CATALOGO_PAISES)} países, "
      f"{sum(1 + len(a) for _, _, a in CATALOGO_PAISES)} grafías declaradas")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  CELDA D — LISTAS DE LIMPIEZA   (la segunda palanca de calidad; edítelas)
# ══════════════════════════════════════════════════════════════════════════
#
#  Tres listas, tres papeles distintos. No las mezcle:
#
#  1. SUFIJOS_EXTRA      se BORRAN del nombre (formas legales).
#  2. GENERICOS_*        NO se borran: se les baja el peso a casi cero. Borrar
#                        "COMERCIALIZADORA" destruye información; ponderarla
#                        a cero hace que la decisión recaiga en el token que
#                        sí identifica. Medido en esta base: IMPORTADORA
#                        (df=737) pesa 5,96 de IDF y ZELECTA (df=490) pesa
#                        6,37 — la marca pesa MENOS que el genérico, porque
#                        la frecuencia de una marca crece con el número de
#                        variantes de la MISMA empresa. Por eso la lista es
#                        un dato declarado y no una inferencia del corpus.
#  3. PRELIMPIEZA        expresiones regulares aplicadas antes de todo.

from record_linkage.matching.normalizadores import LOCALES

# ── 1. Formas legales internacionales que la librería no trae ───────────────
#  Se excluyen a propósito las siglas ambiguas de dos letras (IP, AS, AB, PT,
#  LP, OU, AD, DD): borrarlas destruiría tokens legítimos, y si de verdad son
#  genéricas en su corpus el peso IDF ya las neutraliza.
SUFIJOS_EXTRA = frozenset({
    "BV", "NV", "SARL", "AG", "KG", "OY", "APS", "LDA", "JSC", "OOO", "ZOO",
    "PTE", "PVT", "SDN", "BHD", "DOO", "EOOD", "OOD", "SAC", "EIRL", "ZRT",
    "KFT", "DMCC", "FZE", "FZCO", "WLL", "SAE", "CA", "CV", "UAB", "TOO",
    "ULC", "PJSC", "OJSC", "CJSC", "LLP", "NPO", "COMPAAIA", "COMPAAAA",
})

# ── 2a. Genéricos estructurales: conectores, giro de negocio y geografía ────
#  Cuentan como "ruido" tanto para ponderar como para decidir si la diferencia
#  entre dos nombres es solo ruido.
GENERICOS_ESTRUCTURALES = frozenset({
    # conectores y artículos
    "DE", "DEL", "LA", "EL", "LOS", "LAS", "Y", "YO", "E", "O", "AND", "THE",
    "OF", "FOR", "AL", "DO", "DA", "EN", "A", "POR", "CON", "DI", "DU", "VAN",
    "DER", "DEN",
    # giro de negocio
    "TRADING", "TRADE", "TRADERS", "INTERNATIONAL", "INTERNACIONAL", "GROUP",
    "GRUPO", "HOLDING", "HOLDINGS", "IMPORT", "IMPORTS", "IMPORTACION",
    "IMPORTACIONES", "IMPORTADORA", "IMPORTADOR", "EXPORT", "EXPORTS",
    "EXPORTACION", "EXPORTACIONES", "EXPORTADORA", "EXPORTADOR", "GLOBAL",
    "SERVICE", "SERVICES", "SERVICIO", "SERVICIOS", "DISTRIBUIDORA",
    "DISTRIBUIDOR", "DISTRIBUTION", "DISTRIBUTORS", "DISTRIBUCIONES",
    "INVERSIONES", "INVERSION", "INVESTMENTS", "CORPORACION", "CORPORATION",
    "CORPORATE", "LOGISTICS", "LOGISTICA", "LOGISTIC", "COMERCIALIZADORA",
    "COMERCIAL", "COMMERCIAL", "COMERCIO", "COMMERCE", "INDUSTRIAL",
    "INDUSTRIA", "INDUSTRIAS", "INDUSTRIES", "INDUSTRY", "SOLUTIONS",
    "SOLUCIONES", "WHOLESALE", "WHOLESALER", "SUPPLY", "SUPPLIES",
    "SUMINISTROS", "MARKETING", "SYSTEMS", "SISTEMAS", "ENTERPRISE",
    "ENTERPRISES", "PARTNERS", "VENTURES", "WORLDWIDE", "PRODUCTS",
    "PRODUCTOS", "PRODUCTION", "COMPANY", "CIA", "CONSULTING", "CONSULTORES",
    "CONSULTORIA", "ASESORIAS", "REPRESENTACIONES", "DBA", "GENERAL",
    "MANUFACTURAS", "MANUFACTURING", "CONSTRUCCIONES", "PROYECTOS",
    "TRANSPORTES", "TRANSPORT", "SHIPPING", "CARGO", "FREIGHT", "AGENCIA",
    "AGENCY", "NEGOCIOS", "BUSINESS", "MARKET", "MARKETS", "STORE", "STORES",
    "SHOP", "TECH", "TECHNOLOGY", "TECNOLOGIA",
    # transportistas y referencias de envío: identifican un ENVÍO, no una empresa
    "FEDEX", "DHL", "UPS", "USPS", "TNT", "EXPRESS", "COURIER", "COURRIER",
    "AIRLINES", "AIRWAYS", "AWB", "HAWB", "GUIA", "TRACKING", "ORDER",
    "INVOICE", "REF", "REFERENCIA", "CONSIGNEE", "NOTIFY", "SHIP",
})

#: ¿La geografía es ruido? Con True, "ECOLAB" y "ECOLAB CHILE" se unen — y
#: también "BARRY CALLEBAUT USA" con "BARRY CALLEBAUT CANADA", que son dos
#: sociedades distintas del mismo grupo. Es el modo de error dominante de la
#: banda 0,92–0,96: medido, 4 de 40 asignaciones revisadas en esa banda son
#: empresas hermanas unidas por el nombre de marca. Póngalo en False si
#: necesita resolver a nivel de PERSONA JURÍDICA y no de grupo comercial;
#: perderá, a cambio, los empalmes de sucursal del tipo "NETAFIM QUITO" =
#: "NETAFIM ECUADOR S.A.". Mídalo con la muestra antes de decidir.
GEOGRAFIA_ES_RUIDO = True

# ── 2b. Genéricos DE SECTOR ─────────────────────────────────────────────────
#  Se ponderan a la baja, pero NO cuentan como ruido al comparar: son lo único
#  que separa a "GREENWAY FARMS" de "GREENWAY COFFEE", dos empresas distintas.
#  Si su base es de otro sector, cambie esta lista entera.
GENERICOS_SECTOR = frozenset({
    "FLOWERS", "FLOWER", "FLORAL", "FLORES", "FLOR", "BOUQUET", "BOUQUETS",
    "FARMS", "FARM", "FINCA", "COFFEE", "CAFE", "FRESH", "FRUIT", "FRUITS",
    "FRUTAS", "FOOD", "FOODS", "ALIMENTOS", "ENERGY", "PETROLEUM", "OIL",
    "TEXTIL", "TEXTILE", "TEXTILES", "CONFECCIONES", "PLASTIC", "PLASTICS",
    "QUIMICA", "QUIMICOS", "CHEMICAL", "CHEMICALS", "PHARMA", "FARMACEUTICA",
    "MINERALS", "MINERALES",
})

#: Topónimos: pesan poco SIEMPRE (no identifican a nadie por sí solos), pero
#: solo cuentan como "ruido" al comparar dos nombres si GEOGRAFIA_ES_RUIDO.
GENERICOS_GEOGRAFIA = frozenset({
    "COLOMBIA", "COLOMBIANA", "USA", "US", "AMERICA", "AMERICAN", "AMERICAS",
    "LATIN", "LATINO", "PERU", "MEXICO", "ECUADOR", "PANAMA", "CHILE",
    "BRASIL", "BRAZIL", "COSTA", "RICA", "GUATEMALA", "ESPANA", "SPAIN",
    "CANADA", "EUROPA", "EUROPE", "ASIA", "MIAMI", "FLORIDA", "NEW", "YORK",
    "ANDINA", "NACIONAL", "NATIONAL", "HONDURAS", "SALVADOR", "NICARAGUA",
    "VENEZUELA", "ARGENTINA", "URUGUAY", "PARAGUAY", "BOLIVIA", "DOMINICANA",
    "PUERTO", "ARUBA", "CURACAO", "ITALIA", "ITALY", "FRANCIA", "FRANCE",
    "ALEMANIA", "GERMANY", "HOLLAND", "CHINA", "JAPAN", "JAPON", "KOREA",
    "COREA", "INDIA", "RUSIA", "RUSSIA", "MONTREAL", "TORONTO", "BOGOTA",
    "MEDELLIN", "CALI", "BARRANQUILLA", "CARTAGENA", "QUITO", "GUAYAQUIL",
    "LIMA", "SANTIAGO", "BUENOS", "AIRES", "CARACAS", "MADRID", "BARCELONA",
    "LONDON", "PARIS", "AMSTERDAM", "DUBAI", "HOUSTON", "CHICAGO", "ANGELES",
    "FRANCISCO", "ATLANTA", "DALLAS", "BOSTON", "NORTE", "SOUTH", "NORTH",
    "EAST", "WEST", "CENTRAL", "ORIENTE", "OCCIDENTE",
})

GENERICOS_ESTRUCTURALES = GENERICOS_ESTRUCTURALES | (
    GENERICOS_GEOGRAFIA if GEOGRAFIA_ES_RUIDO else frozenset())
GENERICOS_TODOS = GENERICOS_ESTRUCTURALES | GENERICOS_SECTOR | GENERICOS_GEOGRAFIA

# ── 3. Prelimpieza: (regex, reemplazo), en orden ────────────────────────────
PRELIMPIEZA: tuple[tuple[str, str], ...] = (
    (r"&#\d{2,5};?", " "),                                        # entidades HTML numéricas
    (r"^\s*\(?\s*[0-9][0-9A-Za-z]{0,6}\s*\)?\s*[-–:]\s*", ""),    # código de cliente inicial
    (r"\bC\s*/\s*O\b", " "),                                      # "care of"
    (r"\bA\s*/\s*C\b", " "),                                      # "account of"
)

# ── 4. Locale de normalización de nombres ───────────────────────────────────
#  Combina los diccionarios ES y EN de la librería y añade SUFIJOS_EXTRA.
NOMBRE_LOCALE = "INTL"
LOCALES[NOMBRE_LOCALE] = {
    "sufijos": frozenset(LOCALES["ES"]["sufijos"]) | frozenset(LOCALES["EN"]["sufijos"]) | SUFIJOS_EXTRA,
    "genericos": GENERICOS_TODOS,
}
#: ¿Borrar además los genéricos del nombre? Déjelo en False: la ponderación
#: IDF es estrictamente mejor que el borrado, porque conserva la información
#: para desempatar y no puede dejar un nombre vacío.
QUITAR_GENERICOS_DEL_NOMBRE = False
#: Unir iniciales sueltas: "O.M.G" → "OMG", "U S PLASTICS" → "US PLASTICS".
UNIR_INICIALES = True

print(f"sufijos: {len(LOCALES[NOMBRE_LOCALE]['sufijos'])} · "
      f"genéricos: {len(GENERICOS_TODOS)} "
      f"({len(GENERICOS_ESTRUCTURALES)} estructurales + {len(GENERICOS_SECTOR)} de sector + "
      f"{len(GENERICOS_GEOGRAFIA)} topónimos) · geografía como ruido: {GEOGRAFIA_ES_RUIDO}")

## Ejecución y control de calidad

La Celda E define el motor (no se edita). F ejecuta, G audita y H exporta.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  CELDA E — MOTOR    (definiciones; no requiere edición)
# ══════════════════════════════════════════════════════════════════════════
import html
import re
import time
import unicodedata
from dataclasses import replace
from pathlib import Path

import numpy as np
import pandas as pd
from rapidfuzz import process as rf_process
from rapidfuzz.distance import JaroWinkler
from scipy import sparse

from record_linkage import (
    CampoSpec,
    EsquemaCampos,
    PoliticaFaltante,
    TipoCampo,
    clusters_desde_decisiones,
    evaluar_esquema,
)
from record_linkage.matching.idf import PesosIDF, construir_idf
from record_linkage.matching.motor_bloqueo import BloqueoComponible, LSHTexto
from record_linkage.matching.normalizadores import normalizar_nombre

# ───────────────────────────────────────────────────────────────────────────
#  E.1 · Lectura y saneamiento
# ───────────────────────────────────────────────────────────────────────────


def leer_base(ruta: str | Path) -> pd.DataFrame:
    """Lee CSV/XLSX/Parquet con mensajes accionables si la ruta no existe."""
    p = Path(ruta)
    if not p.exists():
        padre = next((x for x in p.parents if x.exists()), Path("/"))
        vistos = ", ".join(sorted(f.name for f in padre.iterdir())[:15]) or "(vacía)"
        raise FileNotFoundError(
            f"Qué pasó: no existe {p}.\n"
            f"Por qué importa: sin el archivo no hay nada que deduplicar.\n"
            f"Qué hacer: la carpeta más cercana que SÍ existe es {padre} y contiene: "
            f"{vistos}. Corrija RUTA_ENTRADA en la Celda B (cópiela exacta)."
        )
    ext = p.suffix.lower()
    if ext == ".csv" or ext == ".txt":
        return pd.read_csv(p, dtype=str, sep=SEPARADOR_CSV, encoding=ENCODING_CSV)
    if ext in (".xlsx", ".xls", ".xlsm"):
        return pd.read_excel(p, sheet_name=HOJA_EXCEL, dtype=str)
    if ext == ".parquet":
        return pd.read_parquet(p)
    raise ValueError(f"Extensión no soportada: '{ext}'. Use .csv, .xlsx o .parquet.")


def sanear(s: pd.Series) -> pd.Series:
    """Entidades HTML, caracteres de control, espacios raros y PRELIMPIEZA."""
    s = s.astype("string").fillna("")
    s = s.map(lambda t: html.unescape(t) if "&" in t else t)
    s = s.str.replace(r"[   ​-‍﻿]", " ", regex=True)
    s = s.str.replace(r"[\x00-\x1f\x7f-\x9f]", " ", regex=True)
    for patron, reemplazo in PRELIMPIEZA:
        s = s.str.replace(patron, reemplazo, regex=True)
    return s.str.replace(r"\s+", " ", regex=True).str.strip()


def ascii_mayusculas(s: pd.Series) -> pd.Series:
    """Pliega tildes a ASCII y pasa a mayúsculas."""
    return (
        s.astype("string").fillna("").str.normalize("NFKD")
        .str.encode("ascii", errors="ignore").str.decode("ascii").str.upper()
        .str.replace(r"\s+", " ", regex=True).str.strip()
    )


_RE_INICIALES = re.compile(r"(?<![A-Z0-9])((?:[A-Z]\s+){1,6}[A-Z])(?![A-Z0-9])")


def unir_iniciales(s: pd.Series) -> pd.Series:
    """'O.M.G' → 'OMG'; 'U S PLASTICS' → 'US PLASTICS'; 'C.A.' → 'CA'.

    Va SIEMPRE después de plegar tildes: el patrón de puntuación usa semántica
    ASCII (pandas 3 delega la regex en Arrow/RE2), así que aplicarlo antes
    borraría la "É" de "AMÉRICA". Es un error que cuesta el 42 % de la
    canonización de países y no da ningún síntoma visible.
    """
    s = (s.str.replace(r"[^\w\s()]", " ", regex=True)
          .str.replace(r"\s+", " ", regex=True).str.strip())
    if not UNIR_INICIALES:
        return s
    return s.map(lambda t: _RE_INICIALES.sub(lambda m: m.group(1).replace(" ", ""), t))


# ───────────────────────────────────────────────────────────────────────────
#  E.2 · Canonización de país contra el catálogo declarado
# ───────────────────────────────────────────────────────────────────────────


def _clave(texto: str) -> str:
    t = unicodedata.normalize("NFKD", str(texto))
    t = "".join(c for c in t if not unicodedata.combining(c)).upper()
    return re.sub(r"\s+", " ", t).strip()


def indice_paises() -> tuple[dict[str, tuple[str, str]], dict[str, tuple[str, str]]]:
    """(índice exacto, índice sin puntuación). Falla si hay un alias ambiguo."""
    exacto: dict[str, tuple[str, str]] = {}
    laxo: dict[str, tuple[str, str]] = {}
    for iso, canonico, alias in CATALOGO_PAISES:
        for grafia in (canonico, *alias):
            k = _clave(grafia)
            if k in exacto and exacto[k][0] != iso:
                raise ValueError(
                    f"Qué pasó: la grafía '{k}' está declarada en dos países "
                    f"({exacto[k][0]} y {iso}).\nQué hacer: quítela de uno de los dos "
                    "en CATALOGO_PAISES (Celda C)."
                )
            exacto[k] = (iso, canonico)
            kl = re.sub(r"\s+", " ", re.sub(r"[^A-Z0-9 ]", " ", k)).strip()
            laxo.setdefault(kl, (iso, canonico))
    return exacto, laxo


def canonizar_pais(serie: pd.Series) -> pd.DataFrame:
    """País crudo → (ISO3, nombre canónico, método). Nunca falla en silencio."""
    crudo = ascii_mayusculas(sanear(serie))
    exacto, laxo = indice_paises()
    sin_puntuacion = (crudo.str.replace(r"[^A-Z0-9 ]", " ", regex=True)
                      .str.replace(r"\s+", " ", regex=True).str.strip())
    h1 = crudo.map(exacto.get)
    h2 = sin_puntuacion.map(laxo.get)
    es_zf = crudo.str.match("|".join(PATRONES_NO_PAIS)).fillna(False)
    iso3, canonico, metodo = [], [], []
    for a, b, z in zip(h1, h2, es_zf):
        if isinstance(a, tuple):
            iso3.append(a[0]); canonico.append(a[1]); metodo.append("catalogo_exacto")
        elif isinstance(b, tuple):
            iso3.append(b[0]); canonico.append(b[1]); metodo.append("catalogo_laxo")
        elif z:
            iso3.append(ISO_NO_PAIS); canonico.append(ETIQUETA_NO_PAIS); metodo.append("zona_franca")
        else:
            iso3.append("ZZZ"); canonico.append("SIN CLASIFICAR"); metodo.append("sin_clasificar")
    return pd.DataFrame(
        {"PAIS_ISO3": iso3, "PAIS_FINAL": canonico, "PAIS_METODO": metodo,
         "PAIS_NORMALIZADO": crudo.to_numpy()},
        index=serie.index,
    )


def sugerir_alias_pais(sin_clasificar: pd.Series, maximo: int = 25) -> pd.DataFrame:
    """Para lo que no mapeó: el país del catálogo más parecido, como sugerencia.

    Es una ayuda para EDITAR la Celda C, no una asignación automática: los
    pares difíciles (TURQUIA/TURKIYE) no se parecen y los que se parecen
    (GUINEA / GUINEA-BISSAU) son países distintos.
    """
    if sin_clasificar.empty:
        return pd.DataFrame(columns=["valor", "n_filas", "sugerencia", "similitud"])
    conteo = sin_clasificar.value_counts().head(maximo)
    canonicos = [c for _, c, _ in CATALOGO_PAISES]
    filas = []
    for valor, n in conteo.items():
        puntajes = rf_process.cdist([valor], canonicos, scorer=JaroWinkler.normalized_similarity)[0]
        k = int(np.argmax(puntajes))
        filas.append({"valor": valor, "n_filas": int(n),
                      "sugerencia": canonicos[k], "similitud": round(float(puntajes[k]), 3)})
    return pd.DataFrame(filas)


# ───────────────────────────────────────────────────────────────────────────
#  E.3 · Informatividad de tokens (IDF declarado + aprendido)
# ───────────────────────────────────────────────────────────────────────────


def neutralizar_genericos(w: PesosIDF, terminos, peso: float = 1.0) -> PesosIDF:
    """Baja a ``peso`` el IDF de los términos DECLARADOS genéricos.

    Guarda el IDF crudo: la prueba de distintividad (E.4) lo necesita para
    distinguir una palabra de sector de una marca.
    """
    pesos = w.pesos.copy()
    tocados = 0
    for token in terminos:
        i = w.vocabulario.get(str(token).strip().upper())
        if i is not None and pesos[i] > peso:
            pesos[i] = peso
            tocados += 1
    nuevo = replace(w, pesos=pesos, peso_total=np.asarray(w.incidencia @ pesos).ravel())
    object.__setattr__(nuevo, "_pesos_crudos", w.pesos.copy())
    object.__setattr__(nuevo, "_neutralizados", tocados)
    return nuevo


def marcar_informativos(w: PesosIDF, genericos, *, longitud_minima: int,
                        ignorar_numericos: bool) -> np.ndarray:
    """Tokens que SÍ identifican: ni genéricos declarados, ni números, ni siglas.

    Los tokens puramente numéricos ("FEDEX 453927848470") identifican un
    ENVÍO, y su IDF es máximo justo por ser únicos — exactamente al revés de
    lo que conviene. Se excluyen por regla, no por umbral.
    """
    mask = np.zeros(len(w.pesos), dtype=bool)
    gen = {str(g).strip().upper() for g in genericos}
    for token, i in w.vocabulario.items():
        if len(token) < longitud_minima or token in gen:
            continue
        if ignorar_numericos and token.isdigit():
            continue
        mask[i] = True
    return mask


def piso_por_frecuencia(w: PesosIDF, frecuencia_maxima: int) -> float:
    """IDF de un token que aparece ``frecuencia_maxima`` veces en el corpus."""
    if frecuencia_maxima < 1:
        raise ValueError("frecuencia_max_distintivo debe ser >= 1.")
    return float(np.log((1.0 + w.documentos) / (1.0 + frecuencia_maxima)) + 1.0)


def _peso_interseccion(w: PesosIDF, izq: np.ndarray, der: np.ndarray) -> np.ndarray:
    return np.asarray(w.incidencia[izq].multiply(w.incidencia[der]) @ w.pesos).ravel()


def jaccard_idf(w: PesosIDF, izq: np.ndarray, der: np.ndarray) -> np.ndarray:
    """W(A∩B) / W(A∪B): simétrico, no premia que un nombre contenga al otro."""
    inter = _peso_interseccion(w, izq, der)
    union = w.peso_total[izq] + w.peso_total[der] - inter
    with np.errstate(divide="ignore", invalid="ignore"):
        return np.clip(np.where(union > 0, inter / union, 0.0), 0.0, 1.0)


def contencion_idf(w: PesosIDF, izq: np.ndarray, der: np.ndarray) -> np.ndarray:
    """W(A∩B) / min(W(A), W(B)): 1.0 si el nombre corto está dentro del largo."""
    inter = _peso_interseccion(w, izq, der)
    menor = np.minimum(w.peso_total[izq], w.peso_total[der])
    with np.errstate(divide="ignore", invalid="ignore"):
        return np.clip(np.where(menor > 0, inter / menor, 0.0), 0.0, 1.0)


def _matriz_informativa(w: PesosIDF, informativos: np.ndarray):
    inc = getattr(w, "_inc_informativa", None)
    if inc is None:
        inc = (w.incidencia @ sparse.diags(informativos.astype(np.float64))).tocsr()
        inc.eliminate_zeros()
        object.__setattr__(w, "_inc_informativa", inc)
        object.__setattr__(w, "_n_informativos", np.asarray(inc.sum(axis=1)).ravel())
    return inc


def diferencia_informativa(w: PesosIDF, informativos: np.ndarray,
                           izq: np.ndarray, der: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """(|A Δ B|, |A ∩ B|) contando SOLO tokens informativos."""
    inc = _matriz_informativa(w, informativos)
    n = w._n_informativos
    comun = np.asarray(inc[izq].multiply(inc[der]).sum(axis=1)).ravel()
    return n[izq] + n[der] - 2.0 * comun, comun


def max_idf_compartido(w: PesosIDF, informativos: np.ndarray,
                       izq: np.ndarray, der: np.ndarray) -> np.ndarray:
    """IDF CRUDO del token informativo más distintivo que el par comparte."""
    inc_w = getattr(w, "_inc_inf_ponderada", None)
    if inc_w is None:
        crudos = getattr(w, "_pesos_crudos", w.pesos)
        inc_w = w.incidencia.multiply(
            sparse.csr_matrix(np.where(informativos, crudos, 0.0).reshape(1, -1))).tocsr()
        inc_w.eliminate_zeros()
        object.__setattr__(w, "_inc_inf_ponderada", inc_w)
    m = inc_w[izq].multiply(w.incidencia[der]).max(axis=1)
    return np.asarray(m.todense()).ravel() if hasattr(m, "todense") else np.asarray(m).ravel()


# ───────────────────────────────────────────────────────────────────────────
#  E.4 · Comparador de razones sociales
# ───────────────────────────────────────────────────────────────────────────


class SimilitudRazonSocial:
    """Comparador firmado para el motor de `rues-linker`.

    ``sim = alfa · S_tokens + (1 − alfa) · JaroWinkler``, llevada a [−1, 1]
    como exige el contrato de comparadores firmados de la librería.

    ``S_tokens`` es Jaccard ponderado por IDF —simétrico— salvo cuando lo que
    sobra en el nombre largo es puro ruido; ahí usa contención. Las tres
    condiciones para admitir contención salen de fallas medidas:

    1. ``diferencia_informativa ≤ max_diferencia_informativa``
       Sin ella, "MQE" absorbía 214 razones sociales distintas.
    2. ``tokens informativos compartidos ≥ min_informativos_compartidos``
       Sin ella, un nombre sin identidad propia ("INTERNATIONAL") absorbía 22.
    3. ``max IDF crudo compartido ≥ piso``
       Sin ella, "KA DK FLOWERS" y "E FLOWERS" se unían por la palabra
       "FLOWERS", que en esta base aparece en 2.739 nombres.
    """

    name = "razon_social_idf_jw"
    signed = True

    def __init__(self, pesos_idf: PesosIDF, vocabulario, cfg: "Config",
                 genericos_estructurales) -> None:
        self.w = pesos_idf
        self.idx = pd.Index(vocabulario)  # get_indexer: hashtable en C
        self.cfg = cfg
        self.informativos = marcar_informativos(
            pesos_idf, genericos_estructurales,
            longitud_minima=cfg.longitud_minima_token,
            ignorar_numericos=cfg.ignorar_numericos,
        )
        self.piso = (piso_por_frecuencia(pesos_idf, cfg.frecuencia_max_distintivo)
                     if cfg.frecuencia_max_distintivo else 0.0)

    def valid_mask(self, left, right) -> np.ndarray:
        return ((pd.Series(left).astype(str).str.len() > 0).to_numpy()
                & (pd.Series(right).astype(str).str.len() > 0).to_numpy())

    def partes(self, left, right) -> pd.DataFrame:
        """Desglose por par: tokens, Jaro-Winkler, diferencia informativa."""
        sl = pd.Series(left).astype(str).to_numpy()
        sr = pd.Series(right).astype(str).to_numpy()
        jw = rf_process.cpdist(
            sl.tolist(), sr.tolist(), scorer=JaroWinkler.normalized_similarity,
            dtype=np.float64, workers=-1,
            scorer_kwargs={"prefix_weight": self.cfg.prefix_weight},
        )
        ii, jj = self.idx.get_indexer(sl), self.idx.get_indexer(sr)
        conocidos = (ii >= 0) & (jj >= 0)
        tokens = np.zeros(len(sl), dtype=np.float64)
        dif = np.full(len(sl), np.inf)
        contenido = np.zeros(len(sl), dtype=bool)
        if conocidos.any():
            iv, jv = ii[conocidos], jj[conocidos]
            jac = jaccard_idf(self.w, iv, jv)
            con = contencion_idf(self.w, iv, jv)
            d, comun = diferencia_informativa(self.w, self.informativos, iv, jv)
            admite = (d <= self.cfg.max_diferencia_informativa)
            admite &= comun >= self.cfg.min_informativos_compartidos
            if self.piso > 0.0:
                admite &= max_idf_compartido(self.w, self.informativos, iv, jv) >= self.piso
            tokens[conocidos] = np.where(admite, np.maximum(con, jac), jac)
            dif[conocidos] = d
            contenido[conocidos] = admite
        return pd.DataFrame({"sim_tokens": tokens, "jaro_winkler": jw,
                             "dif_informativa": dif, "uso_contencion": contenido})

    def compare(self, left, right) -> np.ndarray:
        if len(left) == 0:
            return np.zeros(0, dtype=np.float64)
        d = self.partes(left, right)
        mezcla = self.cfg.alfa_idf * d["sim_tokens"] + (1 - self.cfg.alfa_idf) * d["jaro_winkler"]
        return np.where(self.valid_mask(left, right), 2.0 * mezcla.to_numpy() - 1.0, 0.0)


# ───────────────────────────────────────────────────────────────────────────
#  E.5 · Cobertura por estrellas (corta el encadenamiento transitivo)
# ───────────────────────────────────────────────────────────────────────────


def _matriz_similitud(comparador, nombres: np.ndarray) -> np.ndarray:
    k = len(nombres)
    a, b = np.meshgrid(np.arange(k), np.arange(k), indexing="ij")
    s = comparador.compare(nombres[a.ravel()], nombres[b.ravel()]).reshape(k, k)
    np.fill_diagonal(s, 1.0)
    return s


def cobertura_estrella(nombres: np.ndarray, etiquetas: np.ndarray, masa: np.ndarray,
                       comparador, cfg: "Config"):
    """Parte cada componente conexa en estrellas y elige el nombre final.

    El clustering por componentes conexas es single-linkage: si a≈b y b≈c, une
    a con c aunque no se parezcan. Sobre razones sociales ese encadenamiento
    es el modo de falla dominante — produjo grupos de 200 empresas distintas.

    Aquí cada grupo queda con un LÍDER y la garantía verificable de que **todo
    miembro está a ≤ (1 − umbral_nombre) de su líder**, que es exactamente lo
    que la correlativa afirma. El líder se elige por cobertura (cubre a más
    miembros; es el medoide operativo), con desempate determinista por masa y
    por nombre. Es la aproximación voraz estándar al star cover.

    Returns:
        (etiquetas nuevas, máscara de líderes, similitud firmada al líder).
    """
    s_min = cfg.sim_minima
    n = len(etiquetas)
    nuevas = np.full(n, -1, dtype=np.int64)
    es_lider = np.zeros(n, dtype=bool)
    sim_lider = np.ones(n, dtype=np.float64)
    siguiente = 0
    orden = np.lexsort((nombres, -masa))
    rango = np.empty(n, dtype=np.int64)
    rango[orden] = np.arange(n)

    for _, posiciones in pd.Series(np.arange(n)).groupby(etiquetas):
        pos = posiciones.to_numpy()
        if len(pos) == 1:
            nuevas[pos] = siguiente; es_lider[pos] = True; siguiente += 1
            continue
        S = _matriz_similitud(comparador, nombres[pos])
        cerca = S >= s_min
        libres = np.ones(len(pos), dtype=bool)
        while libres.any():
            if cfg.regla_nombre_final == "cobertura":
                cobertura = np.where(libres, (cerca & libres[None, :]).sum(axis=1), -1)
                elegido = np.lexsort((rango[pos], -cobertura))[0]
            else:
                elegido = np.flatnonzero(libres)[np.argmin(rango[pos][libres])]
            miembros = np.union1d(np.flatnonzero(libres & cerca[elegido]), [elegido])
            nuevas[pos[miembros]] = siguiente
            sim_lider[pos[miembros]] = S[elegido, miembros]
            es_lider[pos[elegido]] = True
            libres[miembros] = False
            siguiente += 1
    return nuevas, es_lider, sim_lider


# ───────────────────────────────────────────────────────────────────────────
#  E.6 · Preparación y ejecución
# ───────────────────────────────────────────────────────────────────────────


def preparar(df: pd.DataFrame) -> pd.DataFrame:
    """Saneamiento, país canónico y nombre normalizado. Deja constancia."""
    faltan = [c for c in [COL_RAZON_SOCIAL, COL_PAIS] + list(COLS_METRICAS)
              if c not in df.columns]
    if faltan:
        raise KeyError(
            f"Qué pasó: al archivo le faltan las columnas {faltan}.\n"
            f"Por qué importa: son las que declara la Celda B.\n"
            f"Qué hacer: corrija COL_RAZON_SOCIAL / COL_PAIS / COLS_METRICAS. "
            f"El archivo tiene: {list(df.columns)}"
        )
    out = df.copy()
    for c in COLS_METRICAS:
        out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0)

    pais = canonizar_pais(out[COL_PAIS])
    out = pd.concat([out, pais], axis=1)

    crudo = out[COL_RAZON_SOCIAL].astype("string").fillna("")
    limpio = unir_iniciales(ascii_mayusculas(sanear(crudo)))
    out["NOMBRE_NORM"] = normalizar_nombre(
        limpio, locale=NOMBRE_LOCALE, quitar_sufijos=True,
        quitar_genericos=QUITAR_GENERICOS_DEL_NOMBRE,
    )
    n_sin = int((out["NOMBRE_NORM"] == "").sum())
    n_sin_pais = int((out["PAIS_METODO"] == "sin_clasificar").sum())
    print(f"filas leídas            : {len(out):,}")
    print(f"países: {out[COL_PAIS].nunique():,} grafías → "
          f"{out.loc[out.PAIS_METODO != 'sin_clasificar', 'PAIS_FINAL'].nunique():,} canónicos "
          f"| sin clasificar: {n_sin_pais:,} filas")
    print(f"razones sociales: {crudo.nunique():,} grafías → "
          f"{out['NOMBRE_NORM'].nunique():,} normalizadas "
          f"| sin nombre utilizable: {n_sin:,} filas")
    if n_sin_pais:
        print("\n  ATENCIÓN: hay países sin clasificar. Revise SUGERENCIAS_PAIS "
              "y añada los alias que falten a CATALOGO_PAISES (Celda C).")
    return out


def ejecutar(prep: pd.DataFrame, cfg: Config) -> dict:
    """Bloqueo + scoring + clustering + cobertura, partición por país."""
    agregados = {c: (c, "sum") for c in COLS_METRICAS}
    rep = (prep.groupby(["PAIS_ISO3", "PAIS_FINAL", "NOMBRE_NORM"], sort=False)
           .agg(N_FILAS=("NOMBRE_NORM", "size"), **agregados).reset_index())
    rep = rep[rep["NOMBRE_NORM"] != ""].reset_index(drop=True)

    vocabulario = pd.Series(sorted(rep["NOMBRE_NORM"].unique()))
    pesos = construir_idf(vocabulario.reset_index(drop=True))
    pesos = neutralizar_genericos(pesos, GENERICOS_TODOS, peso=cfg.peso_token_generico)
    comparador = SimilitudRazonSocial(pesos, vocabulario.to_numpy(), cfg,
                                      GENERICOS_ESTRUCTURALES)
    print(f"representantes únicos (país, nombre): {len(rep):,}")
    print(f"vocabulario: {len(pesos.vocabulario):,} tokens · "
          f"{getattr(pesos, '_neutralizados', 0)} genéricos neutralizados · "
          f"{int(comparador.informativos.sum()):,} informativos")

    esquema = EsquemaCampos(
        campos=[
            CampoSpec("NOMBRE_NORM", TipoCampo.NOMBRE_EMPRESA, peso=cfg.peso_nombre,
                      comparador=comparador, umbral_concordancia=cfg.sim_minima,
                      params={"quitar_sufijos": False}, locale="EN"),
            CampoSpec("PAIS_FINAL", TipoCampo.CATEGORICO, peso=cfg.peso_pais,
                      veta_discrepancia=True, faltante=PoliticaFaltante.BLOQUEAR),
        ],
        umbral_score=cfg.umbral_score,
        min_concordancias=2,   # nombre Y país; es la definición de "empalme"
        nombre="importadores_razon_social_pais",
    )

    t0 = time.time()
    partes, decisiones, n_candidatos, n_cortes = [], [], 0, 0
    particiones = list(rep.groupby("PAIS_ISO3", sort=True))
    for k, (iso, sub) in enumerate(particiones, 1):
        sub = sub.reset_index(drop=True)
        if len(sub) < 2:
            sub["GRUPO_LOCAL"] = 0; sub["ES_LIDER"] = True; sub["SIM_LIDER"] = 1.0
            partes.append(sub); continue
        bloqueo = BloqueoComponible([
            LSHTexto("NOMBRE_NORM", umbral=cfg.lsh_umbral,
                     permutaciones=cfg.lsh_permutaciones, ngram=cfg.lsh_ngram,
                     max_grupo=cfg.lsh_max_grupo, overflow="connect")
        ])
        res = evaluar_esquema(sub, esquema, bloqueo, max_candidatos=cfg.max_candidatos)
        etiquetas = clusters_desde_decisiones(len(sub), res.decisiones, respetar_vetos=True)
        n_antes = len(np.unique(etiquetas))
        masa = sub["N_FILAS"].to_numpy().astype(np.float64) * 1e9
        if COL_PESO_ECONOMICO and COL_PESO_ECONOMICO in sub.columns:
            masa = masa + sub[COL_PESO_ECONOMICO].to_numpy()
        if cfg.refinar_cohesion:
            etiquetas, lider, sim = cobertura_estrella(
                sub["NOMBRE_NORM"].to_numpy(), etiquetas, masa, comparador, cfg)
            n_cortes += len(np.unique(etiquetas)) - n_antes
        else:
            lider = np.zeros(len(sub), dtype=bool); sim = np.ones(len(sub))
        sub["GRUPO_LOCAL"], sub["ES_LIDER"], sub["SIM_LIDER"] = etiquetas, lider, sim
        partes.append(sub)
        n_candidatos += res.n_candidatos
        if len(res.decisiones):
            decisiones.append(res.decisiones.assign(
                PAIS_ISO3=iso,
                NOMBRE_A=sub["NOMBRE_NORM"].to_numpy()[res.decisiones["i"]],
                NOMBRE_B=sub["NOMBRE_NORM"].to_numpy()[res.decisiones["j"]]))
        if k % 25 == 0 or k == len(particiones):
            print(f"  {k:>3}/{len(particiones)} particiones · "
                  f"{n_candidatos:,} candidatos · {time.time()-t0:.0f}s", end="\r")

    salida = pd.concat(partes, ignore_index=True)
    salida["ID_IMPORTADOR"] = (salida["PAIS_ISO3"] + "-"
                               + salida["GRUPO_LOCAL"].astype(str).str.zfill(6))
    dec = pd.concat(decisiones, ignore_index=True) if decisiones else pd.DataFrame()
    print(f"\ntiempo {time.time()-t0:.0f}s · candidatos {n_candidatos:,} · "
          f"fusiones {int(dec['fusion'].sum()) if len(dec) else 0:,} · "
          f"cortes por cohesión {n_cortes:,}")
    print(f"{len(salida):,} representantes → {salida['ID_IMPORTADOR'].nunique():,} importadores")
    return {"representantes": salida, "decisiones": dec, "comparador": comparador,
            "pesos_idf": pesos, "n_candidatos": n_candidatos, "segundos": time.time() - t0}


# ───────────────────────────────────────────────────────────────────────────
#  E.7 · Entregables
# ───────────────────────────────────────────────────────────────────────────


def construir_entregables(prep: pd.DataFrame, resultado: dict) -> dict[str, pd.DataFrame]:
    """CORRELATIVA, GOLDEN, PAISES y REVISION a partir de la corrida."""
    rep = resultado["representantes"]
    lideres = (rep[rep.ES_LIDER][["ID_IMPORTADOR", "NOMBRE_NORM"]]
               .rename(columns={"NOMBRE_NORM": "NOMBRE_NORM_FINAL"}))
    rep = rep.merge(lideres, on="ID_IMPORTADOR", how="left")

    # Nombre final legible = grafía ORIGINAL más frecuente del nombre líder.
    peso = COL_PESO_ECONOMICO if COL_PESO_ECONOMICO in prep.columns else None
    agg = {"N": (COL_RAZON_SOCIAL, "size")}
    if peso:
        agg["PESO"] = (peso, "sum")
    grafias = (prep.groupby(["PAIS_ISO3", "NOMBRE_NORM", COL_RAZON_SOCIAL], sort=False)
               .agg(**agg).reset_index())
    if not peso:
        grafias["PESO"] = 0.0
    grafias["LARGO"] = grafias[COL_RAZON_SOCIAL].str.len()
    grafias = grafias.sort_values(
        ["PAIS_ISO3", "NOMBRE_NORM", "N", "PESO", "LARGO", COL_RAZON_SOCIAL],
        ascending=[True, True, False, False, False, True], kind="stable")
    modal = (grafias.drop_duplicates(["PAIS_ISO3", "NOMBRE_NORM"])
             [["PAIS_ISO3", "NOMBRE_NORM", COL_RAZON_SOCIAL]]
             .rename(columns={"NOMBRE_NORM": "NOMBRE_NORM_FINAL",
                              COL_RAZON_SOCIAL: "RAZON_SOCIAL_FINAL"}))
    rep = rep.merge(modal, on=["PAIS_ISO3", "NOMBRE_NORM_FINAL"], how="left")
    rep["SIM_AL_FINAL"] = ((rep["SIM_LIDER"] + 1) / 2).round(4)

    corr = prep.merge(
        rep[["PAIS_ISO3", "NOMBRE_NORM", "ID_IMPORTADOR", "NOMBRE_NORM_FINAL",
             "RAZON_SOCIAL_FINAL", "SIM_AL_FINAL", "ES_LIDER"]],
        on=["PAIS_ISO3", "NOMBRE_NORM"], how="left")

    # Nombres que quedan vacíos tras normalizar ("0", "N/A", "-"): cada uno a
    # su propio grupo. Nunca se fusionan entre sí: no hay evidencia de que lo
    # sean, y unirlos inventaría una empresa gigante que no existe.
    sin_nombre = corr["ID_IMPORTADOR"].isna()
    if sin_nombre.any():
        idx = pd.Series(np.arange(int(sin_nombre.sum())), index=corr.index[sin_nombre])
        corr.loc[sin_nombre, "ID_IMPORTADOR"] = "SINNOMBRE-" + idx.astype(str).str.zfill(6)
        corr.loc[sin_nombre, "RAZON_SOCIAL_FINAL"] = corr.loc[sin_nombre, COL_RAZON_SOCIAL]
        corr.loc[sin_nombre, "SIM_AL_FINAL"] = 1.0
        corr.loc[sin_nombre, "ES_LIDER"] = True

    # Identificador que ignora el país (misma empresa en varios destinos).
    llave = rep[["ID_IMPORTADOR", "NOMBRE_NORM_FINAL"]].drop_duplicates()
    global_id = ("G" + pd.Series(llave.groupby("NOMBRE_NORM_FINAL", sort=True).ngroup()
                                 .to_numpy()).astype(str).str.zfill(6))
    mapa = dict(zip(llave["ID_IMPORTADOR"], global_id))
    corr["ID_EMPRESA_GLOBAL"] = corr["ID_IMPORTADOR"].map(mapa).fillna(corr["ID_IMPORTADOR"])

    corr["CAMBIO_NOMBRE"] = corr[COL_RAZON_SOCIAL] != corr["RAZON_SOCIAL_FINAL"]
    corr["CAMBIO_PAIS"] = corr[COL_PAIS] != corr["PAIS_FINAL"]

    columnas = ([COL_RAZON_SOCIAL, COL_PAIS, "RAZON_SOCIAL_FINAL", "PAIS_FINAL",
                 "PAIS_ISO3", "ID_IMPORTADOR", "ID_EMPRESA_GLOBAL", "SIM_AL_FINAL",
                 "CAMBIO_NOMBRE", "CAMBIO_PAIS", "PAIS_METODO", "NOMBRE_NORM"]
                + list(COLS_METRICAS))
    CORRELATIVA = corr[columnas].copy()

    agg_g = {c: (c, "sum") for c in COLS_METRICAS}
    GOLDEN = (CORRELATIVA.groupby(
        ["ID_IMPORTADOR", "ID_EMPRESA_GLOBAL", "RAZON_SOCIAL_FINAL", "PAIS_FINAL", "PAIS_ISO3"],
        sort=False)
        .agg(N_FILAS_ORIGEN=(COL_RAZON_SOCIAL, "size"),
             N_VARIANTES_NOMBRE=(COL_RAZON_SOCIAL, "nunique"),
             SIM_MINIMA=("SIM_AL_FINAL", "min"), **agg_g)
        .reset_index())
    orden = COL_PESO_ECONOMICO if COL_PESO_ECONOMICO in GOLDEN.columns else "N_FILAS_ORIGEN"
    GOLDEN = GOLDEN.sort_values(orden, ascending=False).reset_index(drop=True)

    PAISES = (CORRELATIVA.groupby([COL_PAIS, "PAIS_FINAL", "PAIS_ISO3", "PAIS_METODO"])
              .agg(N_FILAS=("PAIS_FINAL", "size"), **agg_g).reset_index()
              .sort_values(["PAIS_FINAL", "N_FILAS"], ascending=[True, False]))

    revision = GOLDEN.copy()
    motivo = pd.Series("", index=revision.index)
    motivo += np.where(revision["N_VARIANTES_NOMBRE"] >= 15, "grupo_grande;", "")
    motivo += np.where(revision["SIM_MINIMA"] < CFG.umbral_nombre + 0.03, "cohesion_baja;", "")
    motivo += np.where(revision["ID_IMPORTADOR"].str.startswith("SINNOMBRE"),
                       "sin_nombre_utilizable;", "")
    revision["MOTIVO_REVISION"] = motivo
    REVISION = revision[revision.MOTIVO_REVISION != ""].reset_index(drop=True)

    return {"CORRELATIVA": CORRELATIVA, "GOLDEN": GOLDEN,
            "PAISES": PAISES, "REVISION": REVISION}


# ───────────────────────────────────────────────────────────────────────────
#  E.8 · Control de calidad
# ───────────────────────────────────────────────────────────────────────────


def metricas(prep: pd.DataFrame, tablas: dict, resultado: dict, cfg: Config) -> pd.DataFrame:
    """Tabla de indicadores de la corrida. Ninguno requiere verdad de campo."""
    c, g = tablas["CORRELATIVA"], tablas["GOLDEN"]
    peso = COL_PESO_ECONOMICO if COL_PESO_ECONOMICO in c.columns else None
    tam = g["N_VARIANTES_NOMBRE"]
    filas = [
        ("filas de entrada", len(c)),
        ("pares (razón social, país) únicos", len(prep.groupby([COL_RAZON_SOCIAL, COL_PAIS]))),
        ("representantes (país, nombre normalizado)", len(resultado["representantes"])),
        ("importadores finales", g["ID_IMPORTADOR"].nunique()),
        ("empresas finales ignorando país", c["ID_EMPRESA_GLOBAL"].nunique()),
        ("reducción de filas", f"{1 - g['ID_IMPORTADOR'].nunique()/len(c):.1%}"),
        ("filas con nombre reasignado", f"{int(c.CAMBIO_NOMBRE.sum()):,} ({c.CAMBIO_NOMBRE.mean():.1%})"),
        ("filas con país reasignado", f"{int(c.CAMBIO_PAIS.sum()):,} ({c.CAMBIO_PAIS.mean():.1%})"),
        ("grafías de país → canónicos", f"{prep[COL_PAIS].nunique():,} → {c.PAIS_FINAL.nunique():,}"),
        ("grupos con 1 sola variante", f"{int((tam == 1).sum()):,} ({(tam == 1).mean():.1%})"),
        ("grupos con 2–5 variantes", int(tam.between(2, 5).sum())),
        ("grupos con 6–14 variantes", int(tam.between(6, 14).sum())),
        ("grupos con 15+ variantes", int((tam >= 15).sum())),
        ("variantes en el grupo más grande", int(tam.max())),
        ("cohesión mínima del peor grupo", round(float(g["SIM_MINIMA"].min()), 4)),
        ("cohesión media de lo reasignado",
         round(float(c.loc[c.CAMBIO_NOMBRE, "SIM_AL_FINAL"].mean()), 4) if c.CAMBIO_NOMBRE.any() else 1.0),
        ("pares candidatos evaluados", resultado["n_candidatos"]),
        ("segundos de ejecución", round(resultado["segundos"])),
        ("grupos marcados para revisión", len(tablas["REVISION"])),
    ]
    if peso:
        total = c[peso].sum()
        conserv = g[peso].sum()
        filas += [
            (f"{peso} total", f"{total:,.0f}"),
            (f"{peso} conservado en GOLDEN", f"{conserv:,.0f}"),
            ("diferencia (debe ser 0)", f"{total - conserv:,.4f}"),
            ("concentración: top 10 importadores", f"{g[peso].head(10).sum()/total:.2%}"),
            ("concentración: top 100 importadores", f"{g[peso].head(100).sum()/total:.2%}"),
            (f"{peso} en grupos de revisión",
             f"{tablas['REVISION'][peso].sum()/total:.1%}" if len(tablas["REVISION"]) else "0.0%"),
        ]
    return pd.DataFrame(filas, columns=["indicador", "valor"])


def verificar_invariantes(prep: pd.DataFrame, tablas: dict) -> pd.DataFrame:
    """Lo que SIEMPRE debe cumplirse. Si algo falla, el resultado no sirve.

    No son pruebas decorativas: cada una corresponde a un error que este tipo
    de pipeline comete en silencio (perder filas, duplicarlas, mover un
    registro de país, dejar un nombre final vacío, o afirmar una asignación
    por debajo del umbral que se declaró).
    """
    c = tablas["CORRELATIVA"]
    g = tablas["GOLDEN"]
    peso = COL_PESO_ECONOMICO if COL_PESO_ECONOMICO in c.columns else None
    pruebas = [
        ("la correlativa tiene una fila por fila de entrada", len(c) == len(prep)),
        ("ningún importador sin identificador", not c["ID_IMPORTADOR"].isna().any()),
        ("ningún nombre final vacío",
         not (c["RAZON_SOCIAL_FINAL"].isna() | (c["RAZON_SOCIAL_FINAL"].astype(str).str.strip() == "")).any()),
        ("ningún grupo cruza dos países",
         int(c.groupby("ID_IMPORTADOR")["PAIS_ISO3"].nunique().max()) == 1),
        ("toda asignación cumple el umbral declarado",
         bool((c["SIM_AL_FINAL"] >= CFG.umbral_nombre - 1e-9).all())),
        ("cada grupo tiene exactamente un nombre final",
         int(c.groupby("ID_IMPORTADOR")["RAZON_SOCIAL_FINAL"].nunique().max()) == 1),
        ("GOLDEN cubre todos los importadores",
         g["ID_IMPORTADOR"].nunique() == c["ID_IMPORTADOR"].nunique()),
        ("el país final está en el catálogo o marcado",
         bool(c["PAIS_METODO"].isin(["catalogo_exacto", "catalogo_laxo", "zona_franca"]).all())),
    ]
    if peso:
        # Tolerancia RELATIVA, no absoluta: la suma de 200.000 float64 depende
        # del orden de acumulación, y agrupar cambia ese orden. Lo que la
        # invariante debe detectar es una fila perdida o duplicada —cualquiera
        # movería el total muchos órdenes de magnitud por encima de 1e-9
        # relativo—, no el último bit de un punto flotante.
        total_c, total_g = float(c[peso].sum()), float(g[peso].sum())
        relativo = abs(total_c - total_g) / max(abs(total_c), 1.0)
        pruebas.append((f"{peso} se conserva (dif. relativa {relativo:.2e})",
                        bool(relativo <= 1e-9)))
    out = pd.DataFrame(pruebas, columns=["invariante", "cumple"])
    out["estado"] = np.where(out["cumple"], "OK", "FALLA")
    return out


def muestra_para_revision(tablas: dict, cfg: Config) -> pd.DataFrame:
    """Muestra estratificada por banda de similitud, para revisar a mano.

    Es el único camino honesto a una cifra de precisión: sin verdad de campo,
    una métrica interna no dice si el emparejamiento es correcto, solo si es
    consistente. Revise esta muestra y cuente los aciertos por banda.
    """
    c = tablas["CORRELATIVA"]
    asignadas = c[c["CAMBIO_NOMBRE"]].drop_duplicates([COL_RAZON_SOCIAL, "PAIS_ISO3"])
    bandas = [(cfg.umbral_nombre, 0.88), (0.88, 0.92), (0.92, 0.96), (0.96, 1.001)]
    piezas = []
    for lo, hi in bandas:
        s = asignadas[(asignadas.SIM_AL_FINAL >= lo) & (asignadas.SIM_AL_FINAL < hi)]
        if s.empty:
            continue
        take = s.sample(min(cfg.muestra_revision_por_banda, len(s)), random_state=cfg.semilla)
        piezas.append(take.assign(BANDA=f"[{lo:.2f}, {hi:.2f})", POBLACION_BANDA=len(s)))
    if not piezas:
        return pd.DataFrame()
    m = pd.concat(piezas, ignore_index=True)
    m["VEREDICTO_MANUAL"] = ""     # escriba aquí: OK / ERROR / DUDOSO
    return m[["BANDA", "POBLACION_BANDA", "PAIS_FINAL", "SIM_AL_FINAL",
              COL_RAZON_SOCIAL, "RAZON_SOCIAL_FINAL", "VEREDICTO_MANUAL"]]


def recall_del_bloqueo(resultado: dict, cfg: Config) -> pd.DataFrame:
    """PC (pair completeness) exacto por fuerza bruta en las particiones dadas.

    El bloqueo decide qué pares se llegan a comparar; los que no pasan por él
    son invisibles para el resto del pipeline y NO aparecen en ninguna métrica
    posterior. Medido aquí: con 64 permutaciones y umbral 0.35 el bloqueo
    pierde entre el 19 % y el 33 % de los pares verdaderos, sin ningún síntoma.
    Por eso el defecto es 128 permutaciones @ 0.25.
    """
    if not cfg.paises_auditoria_bloqueo:
        return pd.DataFrame()
    rep, comparador = resultado["representantes"], resultado["comparador"]
    filas = []
    for iso in cfg.paises_auditoria_bloqueo:
        sub = rep[rep.PAIS_ISO3 == iso].reset_index(drop=True)
        k = len(sub)
        if k < 2:
            continue
        nombres = sub["NOMBRE_NORM"].to_numpy()
        verdad, paso = set(), 400
        for a in range(0, k, paso):
            ia = np.arange(a, min(a + paso, k))
            for b in range(a, k, paso):
                jb = np.arange(b, min(b + paso, k))
                I, J = np.meshgrid(ia, jb, indexing="ij")
                m = I < J
                if not m.any():
                    continue
                I, J = I[m], J[m]
                ok = comparador.compare(nombres[I], nombres[J]) >= cfg.sim_minima
                verdad.update(zip(I[ok].tolist(), J[ok].tolist()))
        bloqueo = BloqueoComponible([
            LSHTexto("NOMBRE_NORM", umbral=cfg.lsh_umbral,
                     permutaciones=cfg.lsh_permutaciones, ngram=cfg.lsh_ngram,
                     max_grupo=cfg.lsh_max_grupo, overflow="connect")])
        union, _ = bloqueo.pares({"NOMBRE_NORM": nombres})
        candidatos = set(map(tuple, union.tolist()))
        filas.append({
            "pais": iso, "nombres": k, "pares_totales": k * (k - 1) // 2,
            "pares_verdaderos": len(verdad), "candidatos": len(candidatos),
            "PC_recall_bloqueo": round(len(verdad & candidatos) / len(verdad), 4) if verdad else np.nan,
            "RR_reduccion": round(1 - len(candidatos) / (k * (k - 1) / 2), 6),
        })
    return pd.DataFrame(filas)


def sensibilidad_umbral(resultado: dict, cfg: Config,
                        umbrales=(0.80, 0.82, 0.84, 0.86, 0.88, 0.90, 0.92)) -> pd.DataFrame:
    """Cuántas fusiones añade o quita mover el umbral, sin volver a correr todo.

    Usa las decisiones ya calculadas: el número es exacto para el paso de
    scoring (no recalcula la cobertura por estrellas, que solo puede partir
    grupos, nunca unirlos — así que es una COTA SUPERIOR de fusiones).
    """
    dec = resultado["decisiones"]
    if dec.empty:
        return pd.DataFrame()
    sim = ((dec["score"] * (cfg.peso_nombre + cfg.peso_pais) - cfg.peso_pais)
           / cfg.peso_nombre)          # score → similitud firmada de nombre
    sim01 = (sim + 1) / 2
    filas = []
    for u in umbrales:
        n = int((sim01 >= u).sum())
        filas.append({"umbral_nombre": u, "pares_fusionados": n,
                      "vs_defecto": n - int((sim01 >= cfg.umbral_nombre).sum())})
    return pd.DataFrame(filas)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  CELDA F — EJECUCIÓN
# ══════════════════════════════════════════════════════════════════════════
BASE = leer_base(RUTA_ENTRADA)
PREP = preparar(BASE)

#  Si quedaron países sin clasificar, esta tabla propone el alias más cercano.
#  Es una SUGERENCIA para editar la Celda C, no una asignación automática.
SUGERENCIAS_PAIS = sugerir_alias_pais(
    PREP.loc[PREP.PAIS_METODO == "sin_clasificar", "PAIS_NORMALIZADO"])
if len(SUGERENCIAS_PAIS):
    print("\nPaíses sin clasificar (revise y añada a CATALOGO_PAISES):")
    display(SUGERENCIAS_PAIS)

RESULTADO = ejecutar(PREP, CFG)
TABLAS = construir_entregables(PREP, RESULTADO)
TABLAS["CORRELATIVA"].head(10)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  CELDA G — CONTROL DE CALIDAD
# ══════════════════════════════════════════════════════════════════════════
INVARIANTES = verificar_invariantes(PREP, TABLAS)
print("── INVARIANTES ──────────────────────────────────────────────────────")
print(INVARIANTES.to_string(index=False))
if not INVARIANTES["cumple"].all():
    raise AssertionError(
        "Qué pasó: falló al menos una invariante.\n"
        "Por qué importa: el resultado NO es utilizable; alguna fila se perdió, "
        "se duplicó o se asignó por debajo del umbral declarado.\n"
        "Qué hacer: revise la fila marcada FALLA antes de usar nada de esta corrida."
    )

METRICAS = metricas(PREP, TABLAS, RESULTADO, CFG)
print("\n── MÉTRICAS DE LA CORRIDA ───────────────────────────────────────────")
print(METRICAS.to_string(index=False))

SENSIBILIDAD = sensibilidad_umbral(RESULTADO, CFG)
print("\n── SENSIBILIDAD AL UMBRAL (cota superior de fusiones) ───────────────")
print(SENSIBILIDAD.to_string(index=False))

print("\n── RECALL DEL BLOQUEO (fuerza bruta; puede tardar) ──────────────────")
RECALL_BLOQUEO = recall_del_bloqueo(RESULTADO, CFG)
print(RECALL_BLOQUEO.to_string(index=False) if len(RECALL_BLOQUEO)
      else "  (desactivado: CFG.paises_auditoria_bloqueo está vacío)")

MUESTRA = muestra_para_revision(TABLAS, CFG)
print(f"\n── MUESTRA PARA REVISIÓN MANUAL: {len(MUESTRA)} asignaciones ────────")
print("   Llene VEREDICTO_MANUAL con OK / ERROR / DUDOSO y calcule la precisión")
print("   por banda. Es la única cifra de precisión que no se está inventando.")
display(MUESTRA.head(20))

print("\n── LOS 15 GRUPOS MÁS GRANDES (donde vive el riesgo) ─────────────────")
display(TABLAS["GOLDEN"].nlargest(15, "N_VARIANTES_NOMBRE")[
    ["RAZON_SOCIAL_FINAL", "PAIS_FINAL", "N_VARIANTES_NOMBRE", "SIM_MINIMA"]])

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  CELDA H — EXPORTACIÓN
# ══════════════════════════════════════════════════════════════════════════
from record_linkage.exporters.smart import SmartExporter
from record_linkage.pipeline.result import PipelineResult

LIMITE_EXCEL = 1_000_000


def exportar(tablas: dict[str, pd.DataFrame], carpeta: str | Path,
             nombre_base: str = "deduplicacion_importadores") -> list[Path]:
    """XLSX multi-hoja si todo cabe; si no, un CSV.gz por tabla.

    `PipelineResult`/`SmartExporter` neutralizan el texto que empieza por
    `=`, `+`, `-`, `@`, tabulador o salto de línea antes de escribir: una
    razón social puede empezar por `+` (en esta base, una de ellas lo hace) y
    Excel la interpretaría como fórmula.
    """
    carpeta = Path(carpeta)
    carpeta.mkdir(parents=True, exist_ok=True)
    escritos: list[Path] = []
    if all(len(t) <= LIMITE_EXCEL for t in tablas.values()):
        resultado = PipelineResult(work_dir=carpeta, extra=tablas)
        escritos.append(resultado.to_excel(carpeta / f"{nombre_base}.xlsx",
                                           include=tuple(tablas)))
    else:
        exporter = SmartExporter({"output_directory": str(carpeta)})
        for hoja, t in tablas.items():
            escritos.append(Path(exporter.export(t, f"{nombre_base}__{hoja}", format="csv.gz")))
    # La correlativa completa siempre también en Parquet: es el insumo de las
    # siguientes iteraciones y el XLSX pierde tipos.
    ruta_parquet = carpeta / f"{nombre_base}__CORRELATIVA.parquet"
    tablas["CORRELATIVA"].to_parquet(ruta_parquet, index=False)
    escritos.append(ruta_parquet)
    for r in escritos:
        print(f"  escrito: {r}")
    return escritos


A_EXPORTAR = {
    "CORRELATIVA": TABLAS["CORRELATIVA"],
    "GOLDEN": TABLAS["GOLDEN"],
    "PAISES": TABLAS["PAISES"],
    "REVISION": TABLAS["REVISION"],
    "MUESTRA_REVISION": MUESTRA,
    "METRICAS": METRICAS,
    "INVARIANTES": INVARIANTES,
    "SENSIBILIDAD": SENSIBILIDAD,
    "RECALL_BLOQUEO": RECALL_BLOQUEO,
    "PARAMETROS": pd.DataFrame(
        [(k, str(v)) for k, v in CFG.__dict__.items()], columns=["parametro", "valor"]),
}
exportar(A_EXPORTAR, CARPETA_SALIDA)
print("\nEn Colab: descargue la carpeta antes de cerrar la sesión.")

---

## Cómo leer el resultado

| Tabla | Qué es |
|---|---|
| **CORRELATIVA** | Una fila por fila de entrada. `RAZON_SOCIAL_DESTINATARIO` + `PAIS_DESTINO_FINAL` originales junto a `RAZON_SOCIAL_FINAL` + `PAIS_FINAL` asignados, con `SIM_AL_FINAL` (qué tan parecido es el nombre original a su nombre final, 0–1) y las banderas `CAMBIO_NOMBRE` / `CAMBIO_PAIS`. **Es el entregable pedido.** |
| **GOLDEN** | Una fila por importador final: nombre, país, cuántas grafías absorbió, cohesión mínima y las métricas sumadas. |
| **PAISES** | Correlativa del país: cada grafía original con su canónico, el código ISO-3 y el método de asignación. |
| **REVISION** | Los grupos que merecen ojo humano: muchos nombres distintos, cohesión baja o sin nombre utilizable. |
| **MUESTRA_REVISION** | Muestra estratificada por banda de similitud, con una columna vacía para su veredicto. |
| **METRICAS / INVARIANTES / SENSIBILIDAD / RECALL_BLOQUEO / PARAMETROS** | La corrida, auditable. |

Dos identificadores, a propósito:
- **`ID_IMPORTADOR`** respeta el país: la misma empresa en dos destinos son dos filas. Es la definición pedida ("empalme por razón social **y** país").
- **`ID_EMPRESA_GLOBAL`** ignora el país, uniendo grupos cuyo nombre normalizado final coincide **exactamente**. Sirve para "¿cuántas empresas distintas hay?"; no hace emparejamiento difuso entre países.

## Cómo calibrar

1. Corra con los valores por defecto.
2. Abra `MUESTRA_REVISION` y **llene `VEREDICTO_MANUAL`**. Con 40 casos por banda ya se ve dónde se rompe. La precisión por banda es el único número que no se está inventando.
3. Si sobran fusiones en la banda baja → suba `umbral_nombre` (0,86 · 0,88) o suba `alfa_idf`.
4. Si faltan empalmes obvios → baje `umbral_nombre` a 0,82–0,80 y **vuelva a revisar la muestra**: en esta base el tramo 0,80–0,84 cae a ~0,75 de precisión.
5. Si ve un grupo absorbido por una palabra genérica → añádala a `GENERICOS_ESTRUCTURALES` (Celda D). Es más barato y más auditable que mover el umbral.
6. Si ve un consolidador comiéndose a sus clientes (`"‹marca› ‹cliente final›"`) → es el comportamiento esperado con `max_diferencia_informativa = 0` solo cuando lo que sobra es ruido; si igual ocurre, suba `umbral_nombre`.
7. Cambie **un** parámetro a la vez y compare `METRICAS` antes y después. Dos cambios simultáneos no se pueden atribuir.

## Lo que este notebook NO hace

- **No inventa una verdad de campo.** Sin etiquetas humanas no hay precision/recall reales; lo que hay son cohesión, invariantes, recall del bloqueo (ese sí exacto) y una muestra lista para etiquetar.
- **No une empresas entre países.** Por diseño: el país es variable de empalme. `ID_EMPRESA_GLOBAL` solo une coincidencias exactas del nombre normalizado.
- **No resuelve grupos económicos.** `HOLCIM` y `HOLCIM COSTA RICA` se unen si están en el mismo país destino; `ACESCO ECUADOR` y `ACESCO INTERNACIONAL` son entidades jurídicas distintas y el notebook las trata como una sola. Si necesita la distinción, suba `umbral_nombre` y revise.
- **No decide qué hacer con los destinatarios reservados.** Las razones sociales que son `"0"`, `"N/A"` o similares quedan cada una en su propio grupo, marcadas `sin_nombre_utilizable`. En la base de referencia son 58 filas — y el 20,5 % del valor FOB. Ignorarlas silenciosamente sesgaría cualquier ranking.

## Precisión medida sobre la base de referencia

160 asignaciones revisadas a mano (40 por banda), sobre 211.949 registros de
destinatarios de exportación colombianos. "Estricta" cuenta como error los
casos dudosos; "amplia" los cuenta como aciertos.

| Banda de similitud | Asignaciones | Precisión estricta | Precisión amplia |
|---|---:|---:|---:|
| 0,84 – 0,88 | 6.915 | 0,825 | 0,925 |
| 0,88 – 0,92 | 4.223 | 0,950 | 1,000 |
| 0,92 – 0,96 | 3.515 | 0,850 | 0,900 |
| 0,96 – 1,00 | 41.952 | 1,000 | 1,000 |
| **Ponderada** | **56.605** | **0,966** | **0,985** |

**Recall del bloqueo: PC = 1,000** en las tres particiones auditadas por
fuerza bruta (GTM, NLD, DEU), con reducción del espacio de pares de 94–97 %.

Dónde está el error que queda, en orden de frecuencia:
1. **Empresas hermanas del mismo grupo** — `BARRY CALLEBAUT USA` con
   `BARRY CALLEBAUT CANADA`, `LATAM AIRLINES GROUP` con `LATAM AIRLINES
   ECUADOR`, `KOREA EAST-WEST POWER` con `KOREA SOUTH-EAST POWER`. Es la
   consecuencia directa de `GEOGRAFIA_ES_RUIDO = True`. Se apaga con esa
   perilla, al precio de perder los empalmes de sucursal.
2. **Consolidadores de carga** — `X PRESS SHAPEWEAR LL ‹cliente›` cae en el
   grupo de la marca. El destinatario declarado en la DIAN es el
   consolidador, así que el resultado es defendible, pero si le importa el
   cliente final, súbale el umbral y revise `REVISION`.
3. **Nombres degenerados** — razones sociales que tras normalizar quedan en
   una sola palabra corriente. Marcados en `REVISION`.

## Cómo se hizo la medición que respalda los valores por defecto

- **Recall del bloqueo**: fuerza bruta sobre 5 particiones completas (GTM, NLD, CHL, DEU, CRI; 1.683 a 3.298 nombres). Se comparan **todos** los pares y se mide cuántos recupera el LSH.
- **Precisión**: muestra estratificada por banda de similitud, revisada caso por caso. La configuración por defecto se cambió **tres veces** por lo que mostró esa revisión: nombres-imán de un token, palabras de sector haciendo de identidad, y un umbral inicial (0,88) que dejaba fuera empalmes correctos con 92,5 % de precisión.
- **Invariantes**: se verifican en cada corrida y **detienen** el notebook si fallan.

Referencias del método: Fellegi & Sunter (1969), *A Theory for Record Linkage*, JASA 64(328); Spärck Jones (1972), *A statistical interpretation of term specificity*, J. Documentation 28(1); Winkler (1990), *String comparator metrics and enhanced decision rules in the Fellegi-Sunter model*; Christen (2012), *Data Matching*, Springer.